In [ ]:
# import statements

import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt
import cv2

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from scipy.spatial.distance import pdist, squareform

from skimage.draw import polygon
from lmfit import Model, create_params
from skopt import gp_minimize
from scipy.signal import savgol_filter

from skimage.filters.rank import entropy
from skimage.morphology import disk

%matplotlib widget


In [ ]:
# connects to SEM 

PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

In [ ]:
# misc helper function

def take_image(xdim = 256, ydim = 256, stack = 4, HDR = False, scale = 0.5):
    """
    Takes an image of the current location shown on the SEM and returns the image array
    and the pixel dimensions (width and height) in meters.
    """
    #Set to a value < 1 to zoom in on a part of the image - centered # change to less than 1, .5 maybe?

    # phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    return img_array, acq.metadata

In [ ]:
# calibration functions part 1

def calibrate_kernel_size(wd_val_min):
    """
    Returns the kernel size used to apply a blur to the images taken while focusing the SEM.

    Parameter wd_val_min: the minimum working distance (float) that the SEM can image at
    """

    sums = -1
    kernel_size = 5

    phenom.SetSemStigmate(ppi.Position(0, 0)) # sets the stigmate to (0, 0) for sort of optimized stig
    phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52)) # sets magnification to 10000
    phenom.SetSemWD(wd_val_min) # sets the working distance to wd_val_min

    img_array, _ = take_image() # takes image of current location

    while np.round(sums, 4) != 0:
        blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image
        filtered_data = cv2.Canny(blurred_img, 50, 150) # applies a canny filter to the blurred image

        sums = filtered_data.sum() # sums up the pixel values in the filtered image
        
        if np.round(sums, 4) != 0: # if the sum is not 0, add 2 to the kernel size and repeat
            kernel_size = kernel_size + 2

    return kernel_size


In [ ]:
# global variables
wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -0.2 #-1
xst_val_max = 0.2 #1
yst_val_min = -0.2 #-1
yst_val_max = 0.2 #1

size = calibrate_kernel_size(wd_val_min)

In [ ]:
# autocorrelation processing helper funcs

def gradient_2d(data):
    grad_y, grad_x = np.gradient(data)
    combined = np.zeros_like(data)

    for y in range(grad_y.shape[0]):
        for x in range(grad_x.shape[0]):
                combined[x][y] = np.sqrt(grad_x[x][y]**2 + grad_y[x][y]**2)

    return combined


# create slices of the ac based off of slope
def slice_with_slope(data, x_change, y_change):
    data_slice = []
    pts_sampled = []

    if x_change == 0:
        if np.sign(y_change) == 1:
            data_slice = data[data.shape[0] // 2, data.shape[1] // 2:] # typically symmetrical along x or y axis, 
            for y in range(data.shape[1] // 2):                            # so 2d grad doesnt work well
                pts_sampled.append((data.shape[0] // 2, data.shape[1] // 2 + y))
        else:
            data_slice = data[data.shape[0] // 2, :data.shape[1] // 2]
            data_slice = data_slice[::-1]
            for y in range(data.shape[1] // 2):
                pts_sampled.append((data.shape[0] // 2, data.shape[1] // 2 - y))
    
    elif y_change == 0:
        if np.sign(x_change) == 1:
            data_slice = data[data.shape[0] // 2:, data.shape[1] // 2]
            for x in range(data.shape[0] // 2):
                pts_sampled.append((data.shape[0] // 2 + x, data.shape[1] // 2))
        else:
            data_slice = data[:data.shape[0] // 2, data.shape[1] // 2]
            data_slice = data_slice[::-1]
            for x in range(data.shape[0] // 2):
                pts_sampled.append((data.shape[0] // 2 - x, data.shape[1] // 2))

    else:
        center = (data.shape[0] // 2, data.shape[1] // 2)
        x_pos = center[0]
        y_pos = center[1]

        while (x_pos + x_change < data.shape[0] and x_pos + x_change >= 0 and
               y_pos + y_change < data.shape[1] and y_pos + y_change >= 0):
            for x in range(np.abs(x_change)): 
                pts_sampled.append((x_pos, y_pos))
                data_slice.append(data[x_pos, y_pos])
                if x_change > 0:
                    if x < x_change:
                        x_pos += 1
                else:
                    if x < np.abs(x_change):
                        x_pos -= 1

            for y in range(np.abs(y_change)):
                pts_sampled.append((x_pos, y_pos))
                data_slice.append(data[x_pos, y_pos])
                if y_change > 0:
                    if y < y_change:
                        y_pos += 1
                else:
                    if y < np.abs(y_change):
                        y_pos -= 1

    return data_slice, pts_sampled


# use 2nd 2d derivative to find center peak, modify those and then use the modified data or cut off data
def locate_center_peak(data): # grad2
    dir_to_sample = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]

    regions = []
    pts = []
    for pt in dir_to_sample:
        sliced_region, sliced_pts = slice_with_slope(data, pt[0], pt[1])
        regions.append(sliced_region)
        pts.append(sliced_pts)

    max_pts = []
    for index in range(len(regions)):
        max_arg_index = np.argmax(regions[index])
        max_pts.append(pts[index][max_arg_index])

    max_pts = np.array(max_pts)
    x_coords = max_pts[:, 0]
    y_coords = max_pts[:, 1]
    peak_x, peak_y = polygon(x_coords, y_coords)

    return peak_x, peak_y


def interpolate_center_peak(data, peak_x, peak_y): # ac
    for coord in range(peak_x.shape[0]):
        placeholder = [peak_x.min() - 1, peak_y.min() - 1]
        data[peak_x[coord]][peak_y[coord]] = data[placeholder[0], placeholder[1]]


# take sliced pts and check if the peak pts are in there
# if they are, remove the sliced pts and the sliced_region vals that correspond to them or something
# then find max of grad1 slice and the corresponding pt


def find_first_min(data):
    for ind in range(data.shape[0] - 1):
        if data[ind] < data[ind + 1]:
            return ind
        
    return 0


from sklearn.preprocessing import MinMaxScaler


def locate_boundary(gradient_data, peak_x, peak_y): #grad1
    boundary_coords = []

    x_coord_slopes = [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    y_coord_slopes = [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    
    for x_slope in x_coord_slopes:
        for y_slope in y_coord_slopes:
            orig_region, orig_pts = slice_with_slope(gradient_data, x_slope, y_slope)

            indices_to_remove = []
            for pt in range(peak_x.shape[0]):
                if (peak_x[pt], peak_y[pt]) in orig_pts:
                    indices_to_remove.append((peak_x[pt], peak_y[pt]))

            removal_positions = []
            for index, coord in enumerate(orig_pts):
                for removal in indices_to_remove:
                    if coord[0] == removal[0] and coord[1] == removal[1]:
                        removal_positions.append(index)

            no_peak_region = np.delete(orig_region, removal_positions)
            new_region_ind = find_first_min(no_peak_region)
            new_region = no_peak_region[new_region_ind:]


            # # Define the desired feature range
            # scaler = MinMaxScaler(feature_range=(0, new_region.max()))

            # # Fit the scaler to the data and transform it
            # scaled_data = (scaler.fit_transform(new_region.reshape(-1, 1))).ravel()

            # print(f"Original Data:\n{data}")
            # print(f"Scaled Data:\n{scaled_data}")

            window_size = int(new_region.shape[0] * 0.67)
            
            denoised_new_region = savgol_filter(new_region, window_size, 7)
            max_pos = orig_pts[np.argmax(denoised_new_region) + len(indices_to_remove) + new_region_ind]
            boundary_coords.append(max_pos)

    return boundary_coords


def get_ellipse_parameters(ellipse_coords):
    ellipse = np.array(ellipse_coords)
    ellipse_data = cv2.fitEllipse(ellipse) # contour
    (center, axes_lengths, angle) = ellipse_data

    xc, yc = center
    major_axis_len, minor_axis_len = axes_lengths

    major_semi_axis = major_axis_len / 2
    minor_semi_axis = minor_axis_len / 2

    return center, major_semi_axis, minor_semi_axis, angle


def create_ellipse_array(x_center, y_center, a, b, angle_rad, num_points=100):
    """
    Creates a NumPy array of points representing an ellipse.

    Args:
        x_center (float): X-coordinate of the ellipse's center.
        y_center (float): Y-coordinate of the ellipse's center.
        a (float): Length of the semi-major axis.
        b (float): Length of the semi-minor axis.
        angle_rad (float): Rotation angle of the ellipse in radians.
        num_points (int): Number of points to generate for the ellipse.

    Returns:
        numpy.ndarray: A 2D array where each row is an (x, y) coordinate of a point on the ellipse.
    """
    grid = np.linspace(0, 2 * np.pi, num_points)

    # Unrotated ellipse points centered at origin
    x_unrotated = a * np.cos(grid)
    y_unrotated = b * np.sin(grid)

    # Apply rotation
    x_rotated = x_unrotated * np.cos(angle_rad) - y_unrotated * np.sin(angle_rad)
    y_rotated = x_unrotated * np.sin(angle_rad) + y_unrotated * np.cos(angle_rad)

    # Translate to the specified center
    x_coords = x_rotated + x_center
    y_coords = y_rotated + y_center

    return np.array([x_coords, y_coords]).T


In [ ]:
# autocorrelation functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: a Numpy array representing an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def filtered_autocorr(img_array, kernel_size):
    """
    Returns the autocorrelation array for the given image array after applying a blur 
    (given the blurring kernel kernel_size) and a canny filter to img_array.

    Parameter img_array: a Numpy array representing an image
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    # converted_img2 = cv2.cvtColor(img_array, cv2.COLOR_GRAY2BGR)
    # # denoised_img2 = kuwahara(converted_img2, method = 'gaussian', sigma = 1)
    # denoised_img2 = cv2.fastNlMeansDenoisingColored(converted_img2, None, 10, 10, 7, 21)
    # gray_image2 = cv2.cvtColor(denoised_img2, cv2.COLOR_BGR2GRAY)
    # gaussian21 = cv2.GaussianBlur(gray_image2, (3, 3), 0)
    # clahe2 = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
    # equalized_img2 = clahe2.apply(gaussian21)
    # gaussian22 = cv2.GaussianBlur(equalized_img2, (9, 9), 0)
    # filtered_data = cv2.Canny(gaussian22, 50, 150)
    
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    return acdata


def remove_background_autocorr(data):
    ac = data.copy()
    grad1 = gradient_2d(ac) # calculates the amplitudes of first and second derivatives
    grad2 = gradient_2d(grad1)

    peak_x, peak_y = locate_center_peak(grad2)
    interpolate_center_peak(ac, peak_x, peak_y)

    new_grad1 = gradient_2d(ac)

    boundary_coords = locate_boundary(new_grad1, peak_x, peak_y)
    center, major_semi_axis, minor_semi_axis, angle = get_ellipse_parameters(boundary_coords)
    if (round(center[0], -1) == round(ac.shape[0] // 2, -1) and 
        round(center[1], -1) == round(ac.shape[1] // 2, -1)):
        ellipse_points = create_ellipse_array(x_center=center[0], y_center=center[1], a=major_semi_axis, 
                                              b=minor_semi_axis, angle_rad= angle * np.pi / 180, num_points=200)
    else:
        return ac
    
    ellipse_points = np.array(ellipse_points)
    ellipse_x, ellipse_y = polygon(ellipse_points[:, 0], ellipse_points[:, 1])

    if (ellipse_x.min() >= 0 and ellipse_x.min() < ac.shape[0] and
        ellipse_y.min() >= 0 and ellipse_y.min() < ac.shape[1]):
        threshold = ac[ellipse_x.min()][ellipse_y.min()]
    else:
        threshold = ac[0][0]

    blank_image = np.zeros((ac.shape[0], ac.shape[1], 3), dtype=np.uint8) # generates a blank image
    grayscale_ellipse = cv2.cvtColor(blank_image, cv2.COLOR_BGR2GRAY) # generates a grayscale image
    for pt in range(ellipse_x.shape[0]):
        if (ellipse_x[pt] >= 0 and ellipse_x[pt] < ac.shape[0] and
            ellipse_y[pt] >= 0 and ellipse_y[pt] < ac.shape[1]):
            grayscale_ellipse[ellipse_x[pt], ellipse_y[pt]] = 255

    mask = grayscale_ellipse == 0
    ac[mask] = threshold

    for r in range(ac.shape[0]):
        for c in range(ac.shape[1]): # substracts either the threshold or img_array value, whichever is smaller
            ac[r][c] = ac[r][c] - min(threshold, ac[r][c]) # from every point in img_array

    if ac.sum() != 0:
        ac = ac / ac.sum() # normalizes the data

    return ac

In [ ]:
img_array, _ = take_image()

converted_img2 = cv2.cvtColor(img_array, cv2.COLOR_GRAY2BGR)
denoised_img2 = cv2.fastNlMeansDenoisingColored(converted_img2, None, 10, 10, 7, 21)
gray_image2 = cv2.cvtColor(denoised_img2, cv2.COLOR_BGR2GRAY)
gaussian21 = cv2.GaussianBlur(gray_image2, (3, 3), 0)
clahe2 = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
equalized_img2 = clahe2.apply(gaussian21)
gaussian22 = cv2.GaussianBlur(equalized_img2, (9, 9), 0)
filtered2 = cv2.Canny(gaussian22, 50, 150)

plt.figure(figsize=(10,5))
plt.subplot(161), plt.imshow(img_array)
plt.subplot(162), plt.imshow(denoised_img2)
plt.subplot(163), plt.imshow(gaussian21)
plt.subplot(164), plt.imshow(equalized_img2)
plt.subplot(165), plt.imshow(gaussian22)
plt.subplot(166), plt.imshow(filtered2)

In [ ]:
# repositioning funcs

def reposition(kernel_size = size):
    """
    Repositions the SEM so that there is a feature in the center of the imaged location.
    """

    img_array, width, height = take_image() # takes picture of current location
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image array
    filtered = cv2.Canny(blurred_img, 50, 150) # applies a canny filter

    # calculates the distance between each edge pixel in the filtered image and the other edge pixels
    points = [(i, j) for i, row in enumerate(filtered) for j, val in enumerate(row) if val == 255]
    distances = squareform(pdist(points, metric='euclidean'))
    total_distances = [sum(row) for row in distances]

    # selects the pixel with the smalleset distance between other pixels
    new_center = (points[np.argmin(total_distances)][1], points[np.argmin(total_distances)][0])

    # moves to the new center
    center = (img_array.shape[0] // 2, img_array.shape[1] // 2)
    px_distance = [center[0] - new_center[0], center[1] - new_center[1]]
    m_distance = [px_distance[0] * width, px_distance[1] * height]
    phenom.MoveBy(-1 * m_distance[0], m_distance[1])


def reposition_v2():
    img_array, meta = take_image() # takes picture of current location
    entropy_filtered = entropy(img_array, disk(5)) # calculates the entropy

    flat_index = np.argmax(entropy_filtered)
    y, x = np.unravel_index(flat_index, entropy_filtered.shape)
    new_center = (x, y)

    # moves to the new center
    center = (img_array.shape[0] // 2, img_array.shape[1] // 2)
    px_distance = [center[0] - new_center[0], center[1] - new_center[1]]
    m_distance = [px_distance[0] * meta.pixelSize.width, px_distance[1] * meta.pixelSize.height]
    phenom.MoveBy(-1 * m_distance[0], m_distance[1])

    # fig, (ax0, ax1) = plt.subplots(ncols=2, sharex=True, sharey=True)
    # img0 = ax0.imshow(img_array, cmap=plt.cm.gray)
    # ax0.set_title("Image")
    # ax0.axis("off")
    # fig.colorbar(img0, ax=ax0)
    # img1 = ax1.imshow(entropy_filtered, cmap='gray')
    # ax1.set_title("Entropy")
    # ax1.axis("off")
    # fig.colorbar(img1, ax=ax1)
    # fig.tight_layout()
    # plt.show()

    # specific_keypoint = cv2.KeyPoint(x=int(x), y=int(y), size=10, angle=0, response=0, octave=0, class_id=0)
    # keypoints_to_draw = [specific_keypoint]

    # img3 = cv2.drawKeypoints(img_array, keypoints_to_draw, None, color=(0,255,0), flags=0)
    # plt.figure()
    # plt.imshow(img3), plt.show()


def pixelposition(img, meta, xix, yix, ratio):
    """
    Obtain position on stage for pixel with index xix, yix in image img
    note that ratio > 1 the ratio HFW_old/HFW_new still has to be set
    Returns:
        Position object

    Afterwards: 
        phenom.MoveTo(pixelposition(img, meta, xix, yix))
        phenom.SetHFW(phenom.GetHFW()/ratio)
    """
    xpix = xix
    ypix = img.shape[1] - yix

    x, y = meta.position.x, meta.position.y

    x_size = img.shape[0]/ratio
    y_size = img.shape[1]/ratio

    xnew = x + (xpix + x_size//2 - img.shape[0] // 2) * meta.pixelSize.width
    ynew = y + (ypix - y_size//2 - img.shape[1] // 2) * meta.pixelSize.height

    return ppi.Position(xnew, ynew)


def find_ac_position(img, meta, x_size = 64, y_size = 64):
    acdata = np.zeros((img.shape[0]-x_size+1, img.shape[1]-y_size+1, x_size, y_size))

    for i in range(0, img.shape[0]-x_size+1):
        for j in range(0, img.shape[1]-y_size+1):
            window = img[i:i+x_size,j:j+y_size] # each individual window
            # result = filtered_autocorr(window, 9)
            result = auto_correlation(window)
            acdata[i, j] = result

    xix, yix = np.unravel_index(np.argmin(acdata.sum(axis=(2,3))), acdata.shape[:2])
    ratio = img.shape[0] / x_size
    
    return (xix, yix), pixelposition(img, meta, xix, yix, ratio), ratio, acdata


In [ ]:
img, meta = take_image()
(xix, yix), optpos, ratio, acdata = find_ac_position(img, meta)
phenom.MoveTo(optpos)
phenom.SetHFW(phenom.GetHFW()/ratio)
# newimg, newmeta = take_image()

# plt.figure()
# plt.imshow(img)

# plt.plot([xix], [yix], 'ro')

# plt.figure()
# plt.imshow(img[yix:yix + 64, xix:xix + 64])

# plt.figure()
# plt.imshow(newimg)

# plt.figure()
# plt.imshow(auto_correlation(img[yix:yix + 64, xix:xix + 64]))

# plt.figure()
# plt.imshow(auto_correlation(newimg))

In [ ]:
# fitting functions

# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


def incoherent_zernike_psf(N=256, defocus = None, xstig = None, ystig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if xstig is not None:
        zernike_coeffs[6] = xstig

    if ystig is not None:
        zernike_coeffs[4] = ystig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, xst, yst):
    model = Model(incoherent_zernike_psf)

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True, min = 0.1, max=5),
                         xstig = dict(value = xst, vary = True, min=-4, max=4),
                         ystig = dict(value = yst, vary = True, min=-4, max=4))

    result = model.fit(ac, params = pars) # uses the above parameters and  model to fit the data

    return result

In [ ]:
# optimization functions

def optimum_wd(wd, kernel_size = size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the 
    center pixel given wd, a list containing a working distance.
    This is the objective function to minimize the working distance.

    Parameter wd: a tuple containing an int or float specifying the working distance 
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    phenom.SetSemWD(wd[0]) # sets the working distance to wd[0]

    if image_counter[0] % 10 == 0:
        phenom.SemAutoContrastBrightness()
    img_array, _ = take_image() # takes an image of the current location
    image_counter[0] += 1

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the autocorrelation for the filtered image
    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1]) # sums together the four center pixels
    
    return sums * (-1) # returns the negative of the sum


def auto_focus(wd_val_min, wd_val_max):
    """
    Determines the working distance corresponding to a focused image using bayesian optimization. 
    Sets the working distance to the value corresponding to a focused image after minimization.
    Returns the optimal working distance.

    Parameter wd_val_min: the minimum working distance (float) that is considered when minimizing
    Parameter wd_val_max: the maximum working distance (float) that is considered when minimizing
    """

    res = gp_minimize(optimum_wd, # calls scikit's bayesian optimization 
                      [(wd_val_min, wd_val_max)], 
                      acq_func = "EI", 
                      n_calls = 10, 
                      n_initial_points = 5, 
                      verbose = False, 
                      random_state = 1234) 

    phenom.SetSemWD(res.x[0]) # sets the working distance to the optimum
    return res.x[0]


def optimum_stig(st):
    """ 
    Returns the astigmatism coefficient of a Zernike polynomial fitted to the autocorrelation data
    at the given stigmate st.

    Parameter st: a tuple (x, y) for the x-stigmate and y-stigmate
    """

    xst, yst = st
    phenom.SetSemStigmate(ppi.Position(xst, yst))

    if image_counter[0] % 10 == 0:
        phenom.SemAutoContrastBrightness()
    img_array, _ = take_image() # takes an image of the current location
    image_counter[0] += 1

    acdata = auto_correlation(img_array) # calculates the autocorrelation for the image
    filtered_acdata = remove_background_autocorr(acdata) # removes the background of the autocorrelation

    result = incoh_psf_fit(filtered_acdata, 1, 1, 1) # fits the data

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)
    plt.subplot(1, 3, 2)
    plt.imshow(filtered_acdata)
    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return np.sqrt(result.best_values['xstig'] ** 2 + result.best_values['ystig'] ** 2)


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    """
    Determines the stigmate values corresponding to a focused image using bayesian optimization. 
    Sets the working distance to the value corresponding to a focused image after minimization.

    Parameter xst_val_min: the minimum x-stigmate (float) that is considered when minimizing
    Parameter xst_val_max: the maximum x-stigmate distance (float) that is considered when minimizing
    Parameter yst_val_min: the minimum y-stigmate distance (float) that is considered when minimizing
    Parameter yst_val_max: the maximum y-stigmate distance (float) that is considered when minimizing
    """

    res = gp_minimize(optimum_stig, 
                      [(xst_val_min, xst_val_max), (yst_val_min, yst_val_max)], 
                      acq_func = "EI", 
                      n_calls = 30, 
                      n_initial_points = 5, 
                      verbose = False, 
                      random_state = 1234) 

    print(res.x[0], res.x[1])
    phenom.SetSemStigmate(ppi.Position(res.x[0], res.x[1]))

    return res.x[0], res.x[1]


In [ ]:
image_counter = [0]
best_wd = auto_focus(wd_val_min, wd_val_max)

In [ ]:
# phenom.SetSemWD(best_wd - 0.0003) # sets the wd to slightly not optimal
# phenom.SetHFW(ppi.MagnificationToFieldWidth(100000, 0.52))

# do the stig optimization thingy 
# set wd to blurry stuff

image_counter = [0]
x, y = auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)

In [ ]:
image_counter = [0]

def thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
    curr = auto_focus(wd_val_min, wd_val_max)
    reposition_v2()

    best_wd = 0

    for magnification in [20000, 40000]:
        phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))
        best_wd = auto_focus(curr - (1 / 1000), curr + (1 / 1000))
        # reposition_v2()

    phenom.SetSemWD(best_wd - 0.00015) # sets the wd to slightly not optimal
    phenom.SetHFW(ppi.MagnificationToFieldWidth(100000, 0.52))
    x, y = auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)
    phenom.SetSemWD(best_wd)

    print('stig', x, y)

    for r in range(2):
        phenom.SetHFW(ppi.MagnificationToFieldWidth(80000, 0.52))
        best_wd = auto_focus(curr - (1 / 1000), curr + (1 / 1000))

        phenom.SetSemWD(best_wd - 0.000015) # sets the wd to slightly not optimal
        phenom.SetHFW(ppi.MagnificationToFieldWidth(100000, 0.52))
        auto_stig(x - (0.1 / 20), x + (0.1 / 20), y - (0.1 / 20), y + (0.1 / 20))
    
    phenom.SetSemWD(best_wd)

# thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)

In [ ]:
# test this first and check if it works at all the locations i need it to (pull from loc_coords below)
# need to keep two sep positions for wd and stig optimization?

image_counter = [0]

def thingy2(wd_val_min, wd_val_max):
    phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
    curr = auto_focus(wd_val_min, wd_val_max)
    reposition_v2()

    best_wd = 0

    for magnification in [20000, 40000, 60000, 80000, 120000]:
        phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))
        best_wd = auto_focus(curr - (1 / 1000), curr + (1 / 1000))
        # reposition_v2()
    
    phenom.SetSemWD(best_wd)
    return best_wd

thingy2(wd_val_min, wd_val_max)

In [ ]:
# 40k, 60k, 80k, 100k, 120k, 140k, 160k magnifications

# start at __k, then do the stig reposition thing, then take image, then go back to __k, move, repeat

magnifications = [40000, 60000, 80000, 100000, 120000]
wd_change = np.linspace(0, 0.3, 31)
loc_nums = np.linspace(0, 48, 49)

loc_x_coords = np.linspace(-0.00793491, 0.00772189, 7) # border 1, # border 2, 7
loc_y_coords = np.linspace(0.00575148, -0.00511243, 7) # border 1, # border 2, 7
loc_coords = []
for x in range(loc_x_coords.shape[0]):
    for y in range(loc_y_coords.shape[0]):
        loc_coords.append([loc_x_coords[x], loc_y_coords[y]])

phenom.SetSemStigmate(ppi.Position(0, 0)) # mostly optimized stig

for loc in loc_nums:
    phenom.MoveTo(ppi.Position(loc_coords[int(loc)][0], loc_coords[int(loc)][1]))
    del size[0]
    size.append(calibrate_kernel_size(wd_val_min))
    print(size)

    for magnification in magnifications:
        best_wd = thingy2(wd_val_min, wd_val_max) # autofocuses
        phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52)) # sets magnification 

        # ie 10k to 120k through 20k increments? and then take pics at 40k 80k 100k 120k

        for change in wd_change:
            phenom.SetSemWD(best_wd + (change / 1000))
            img, meta = take_image()
            img_array = np.array(img)
            np.save("mag" + str(magnification) + "\\loc" + str(int(loc)) + ',' + str(change), img_array)


